In [ ]:
import pandas as pd
import requests
import io

# Definir la URL que te da el ejercicio
url = "https://www.iposcoop.com/ipos-recently-filed/"

# Leer la tabla HTML aplicando el truco del documento para evitar el error OSError
# pd.read_html devuelve una lista con todas las tablas de la página. La principal suele ser la [0].
tablas = pd.read_html(io.StringIO(respuesta.text))
df = tablas[0]

# Filtrar las filas para quedarte solo con las OPV canceladas
df_retiradas = df[df['Expected To Trade'] == 'Withdrawn']

# Comprobación rápida: el ejercicio dice que deberías ver unas 32 entradas
print(f"Total de empresas retiradas: {len(df_retiradas)}")

Total de empresas retiradas: 34


In [ ]:
import numpy as np

# Hacemos una copia para trabajar seguros sobre los datos filtrados y evitar warnings
df_retiradas = df_retiradas.copy()

# Definimos las condiciones respetando la prioridad
condiciones = [
    df_retiradas['Company'].str.contains('Technologies', na=False),
    df_retiradas['Company'].str.contains('Acquisition Corp|Acquisition Corporation|Corp', na=False),
    df_retiradas['Company'].str.contains('Inc|Incorporated', na=False),
    df_retiradas['Company'].str.contains('Group', na=False),
    df_retiradas['Company'].str.contains('Ltd|Limited', na=False),
    df_retiradas['Company'].str.contains('Holdings|Holding', na=False)
]

#Definimos las categorías que corresponden a cada condición
resultados = [
    'Technologies',
    'Acquisition Corp',
    'Inc.',
    'Group',
    'Limited',
    'Holdings'
]

# Asignamos la columna de golpe. Si una fila no cumple ninguna, cae en el default.
df_retiradas['Company Type'] = np.select(condiciones, resultados, default='Other')

df_retiradas.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
17,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O,Group
20,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O,Group
32,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O,Limited
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Technologies
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings


In [ ]:
import numpy as np

# Función para limpiar el texto financiero y convertirlo a número decimal (float)
def extraer_precio(valor):
    # Convertimos a string y eliminamos espacios en blanco a los lados por seguridad
    valor_str = str(valor).strip()

    # Si detectamos un guion, vacío o valores nulos de Pandas, retornamos un valor nulo matemático (NaN)
    if valor_str == '-' or valor_str.lower() in ['nan', 'none', '']:
        return np.nan

    try:
        # Quitamos el símbolo de dólar y posibles comas de miles, y transformamos a decimal
        return float(valor_str.replace('$', '').replace(',', ''))
    except ValueError:
        # Si la celda tiene un formato extraño que no se puede convertir, la tratamos como nula
        return np.nan

# Aplicamos la función a las columnas de precios
df_retiradas['Min_num'] = df_retiradas['Price Low'].apply(extraer_precio)
df_retiradas['Max_num'] = df_retiradas['Price High'].apply(extraer_precio)

# Calculamos el campo 'Avg_price' sumando ambos extremos y dividiendo entre 2.
# Si en alguna fila 'Min_num' o 'Max_num' resultó ser NaN, la operación entera dará NaN automáticamente.
df_retiradas['Avg_price'] = (df_retiradas['Min_num'] + df_retiradas['Max_num']) / 2
df_retiradas.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Min_num,Max_num,Avg_price
17,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O,Group,4.0,4.0,4.0
20,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O,Group,4.0,5.0,4.5
32,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O,Limited,4.5,5.5,5.0
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Technologies,NaN,NaN,NaN
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings,4.0,5.0,4.5


In [ ]:
# Primero, limpiamos las columnas de acciones y volumen estimado para asegurar que sean numéricas.
# Podemos reutilizar la función 'extraer_precio' que creamos en el paso anterior,
# ya que también elimina el símbolo '$', las comas y maneja los guiones.


df_retiradas['Shares_num'] = df_retiradas['Shares (millions)'].apply(extraer_precio)
df_retiradas['Est_Vol_num'] = df_retiradas['Est $ Vol (millions)'].apply(extraer_precio)

# Calculamos la primera opción: Acciones multiplicadas por el precio promedio
valor_calculado = df_retiradas['Shares_num'] * df_retiradas['Avg_price']

# Creamos la columna final 'Shares_offered_value'.
# El método .fillna() revisa fila por fila:
# Si 'valor_calculado' tiene un número, lo deja intacto.
# Si 'valor_calculado' es nulo (NaN), rellena ese hueco con el valor de 'Est_Vol_num'.
df_retiradas['Shares_offered_value'] = valor_calculado.fillna(df_retiradas['Est_Vol_num'])
df_retiradas.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Min_num,Max_num,Avg_price,Shares_num,Est_Vol_num,Shares_offered_value
17,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O,Group,4.0,4.0,4.0,1.80,7.20,7.200
20,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O,Group,4.0,5.0,4.5,3.75,16.88,16.875
32,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O,Limited,4.5,5.5,5.0,4.40,22.00,22.000
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Technologies,NaN,NaN,NaN,0.00,100.00,100.000
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings,4.0,5.0,4.5,2.00,9.00,9.000


In [ ]:
# Agrupamos por 'Company Type', sumamos el valor ofrecido y ordenamos de mayor a menor
resultado_final = df_retiradas.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)

# Imprimimos el resultado para ver qué categoría quedó en primer lugar y con qué valor
print(resultado_final)
print(f"\nLa categoría con mayor valor retirado es '{resultado_final.index[0]}' con {resultado_final.iloc[0]} millones.")

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             219.2500
Technologies        184.9000
Group                56.5750
Name: Shares_offered_value, dtype: float64

La categoría con mayor valor retirado es 'Acquisition Corp' con 499.985 millones.


2. What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?

In [ ]:
import pandas as pd
import numpy as np
import requests
import io

#Cargar datos de las IPOs de 2025 usando el mismo truco
url = "https://www.iposcoop.com/2025-pricings/"

# Actualizamos la variable 'respuesta' para que contenga el contenido de la URL actual
respuesta = requests.get(url)
df_ipos = pd.read_html(io.StringIO(respuesta.text))[0]

# Preparar y filtrar los datos:
# Convertimos la columna de fecha a formato datetime de Pandas para poder hacer comparaciones.
df_ipos['Date'] = pd.to_datetime(df_ipos['Offer Date'])

# Limpiamos la columna de rentabilidad (Return) quitando el '%' y pasándola a decimal/flotante.
# Si ya es numérica, esto no afectará, pero aseguramos evitar errores de formato string.
df_ipos['Return_num'] = df_ipos['Return'].astype(str).str.replace('%', '').astype(float)

# Aplicamos los dos filtros solicitados:
# a) Salieron a bolsa antes del 1 de septiembre de 2025.
# b) Tienen una rentabilidad distinta de 0 (excluir valores inactivos/planos).
df_filtrado = df_ipos[(df_ipos['Date'] < '2025-09-01') & (df_ipos['Return_num'] != 0.0)]

# Extraemos los símbolos (tickers) como una lista limpia para pasársela a Yahoo Finance.
tickers = df_filtrado['Symbol'].dropna().tolist()

df_filtrado.head()

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating,Date,Return_num
68,"Picard Medical, Inc.",PMI,Health Care,8/29/2025,4.3,$4.00,$4.59,$1.68,-58.00%,S/O,2025-08-29,-58.00
69,GrowHub Limited (The),TGHL,Technology,8/28/2025,3.7,$4.00,$3.38,$0.40,-90.00%,S/O,2025-08-28,-90.00
70,TryHard Holdings Limited,THH,Consumer Services,8/28/2025,1.5,$4.00,$6.02,$29.47,636.75%,S/O,2025-08-28,636.75
71,Curanex Pharmaceuticals,CURX,Health Care,8/26/2025,3.8,$4.00,$4.02,$0.37,-90.75%,S/O,2025-08-26,-90.75
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,8/21/2025,40.0,$10.00,$10.12,$10.12,1.20%,S/O,2025-08-21,1.20


In [ ]:
import yfinance as yf

# Bajamos el historial de todos los tickers filtrados en un solo bloque.
# Ponemos fechas amplias para asegurar que el cálculo de shift(252) tenga historial suficiente.
data = yf.download(tickers, start="2025-01-01", end="2026-09-15")

# Extraemos solo la tabla de precios de cierre.
precios_cierre = data['Close']

# Creamos una lista para ir guardando los datos procesados de cada empresa
resultados = []

# Iteramos sobre cada columna (cada ticker) para calcular sus métricas individualmente
for ticker in precios_cierre.columns:
    # Seleccionamos la columna del ticker, eliminamos nulos (días sin cotizar) y la convertimos en un DataFrame
    df_ticker = precios_cierre[[ticker]].dropna().copy()

    # Aplanamos el nombre de la columna para evitar errores con índices múltiples y la llamamos 'Close'
    df_ticker.columns = ['Close']

    # Crecimiento tras aproximadamente un año (252 días hábiles de bolsa)
    df_ticker['growth_252d'] = df_ticker['Close'] / df_ticker['Close'].shift(252)

    # Volatilidad anualizada: Desviación estándar de los últimos 30 días multiplicada por raíz de 252
    df_ticker['volatility'] = df_ticker['Close'].rolling(30).std() * np.sqrt(252)

    # Fórmula: (Crecimiento - Tasa libre de riesgo) / Volatilidad. Tasa libre de riesgo = 5% (0.05).
    df_ticker['Sharpe'] = (df_ticker['growth_252d'] - 0.05) / df_ticker['volatility']

    # Añadimos el nombre de la empresa para identificar la fila, sacamos la fecha del índice a una columna
    # y guardamos este pequeño DataFrame en nuestra lista.
    df_ticker['Ticker'] = ticker
    df_ticker = df_ticker.reset_index()
    resultados.append(df_ticker)

# Unimos todos los cálculos individuales en un único DataFrame principal consolidado
stocks_df = pd.concat(resultados, ignore_index=True)

/tmp/ipykernel_414/1945276683.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2025-01-01", end="2026-09-15")
[*********************100%***********************]  146 of 146 completed
ERROR:yfinance:
14 Failed downloads:
ERROR:yfinance:['EPWK', 'CEPT', 'SDM', 'SKBL', 'MJID', 'CAEP', 'MTSR', 'EMPG', 'AGH', 'MCTR', 'PTNM', 'TBH', 'AHL', 'WGRX']: YFTzMissingError('possibly delisted; no timezone found')


In [ ]:
# Filtramos el DataFrame para obtener los datos al 11 de septiembre de 2026
sharpe_ratio_2026_09_11 = stocks_df[stocks_df['Date'] == '2026-09-11']

# Calculamos la mediana del Sharpe ratio
median_sharpe_ratio = sharpe_ratio_2026_09_11['Sharpe'].median()

print(f"La mediana del Sharpe ratio al 11 de septiembre de 2026 es: {median_sharpe_ratio:.4f}")

La mediana del Sharpe ratio al 11 de septiembre de 2026 es: 0.0501


What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?

In [ ]:
# Trabajamos sobre el dataset de la pregunta 2
df_q3 = stocks_df.copy()

# Ingeniería de características: Calcular el crecimiento futuro de 1 a 12 meses
# Usamos un bucle para crear las 12 columnas automáticamente (1 mes = 21 días)
for mes in range(1, 13):
    dias = mes * 21
    nombre_columna = f'future_growth_{mes}_m'
    df_q3[nombre_columna] = df_q3.groupby('Ticker')['Close'].shift(-dias) / df_q3['Close']

# Identificar puntos de entrada:
# Buscamos la fecha más antigua (el día de la IPO) para cada empresa.
primeros_dias = df_q3.groupby('Ticker')['Date'].min().reset_index()
:
# Hacemos un inner join para quedarnos ÚNICAMENTE con la fila del primer día de cotización.
df_ipo_days = pd.merge(df_q3, primeros_dias, on=['Ticker', 'Date'], how='inner')

# 5. Análisis de la mediana:
# Creamos una lista solo con los nombres de las 12 columnas nuevas
columnas_crecimiento = [f'future_growth_{mes}_m' for mes in range(1, 13)]

# Calculamos la mediana de esas columnas
medianas = df_ipo_days[columnas_crecimiento].median()

# Imprimimos los resultados para analizar la tendencia
print("Medianas de crecimiento por mes de tenencia:")
print(medianas)

# Identificamos automáticamente el ganador
mejor_mes = medianas.idxmax()
mejor_valor = medianas.max()

print(f"\nEl período óptimo para mantener la acción es '{mejor_mes}' con una rentabilidad mediana de {mejor_valor:.4f}")

Medianas de crecimiento por mes de tenencia:
future_growth_1_m     0.935351
future_growth_2_m     0.892958
future_growth_3_m     0.827160
future_growth_4_m     0.730400
future_growth_5_m     0.690909
future_growth_6_m     0.726000
future_growth_7_m     0.662000
future_growth_8_m     0.603508
future_growth_9_m     0.580294
future_growth_10_m    0.533333
future_growth_11_m    0.481811
future_growth_12_m    0.491838
dtype: float64

El período óptimo para mantener la acción es 'future_growth_1_m' con una rentabilidad mediana de 0.9354


4. What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?

In [ ]:
import gdown
import pandas as pd

# Descargamos el archivo parquet desde Google Drive
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

# Leemos el archivo usando el motor pyarrow
df_estrategia = pd.read_parquet("data.parquet", engine="pyarrow")

# Aseguramos que la columna 'Date' sea de tipo datetime para poder filtrar correctamente
df_estrategia['Date'] = pd.to_datetime(df_estrategia['Date'])

# Seleccionamos las fechas entre 2000-01-01 y 2025-06-01, y donde el RSI sea menor a 30
df_filtrado_rsi = df_estrategia[
    (df_estrategia['Date'] >= '2000-01-01') &
    (df_estrategia['Date'] <= '2025-06-01') &
    (df_estrategia['rsi'] < 30)
]

# Aplicamos la fórmula matemática proporcionada por el ejercicio
ingreso_neto_total = 1000 * (df_filtrado_rsi['growth_future_30d'] - 1).sum()

# Convertimos el resultado
print(f"El ingreso neto total obtenido es de: ${ingreso_neto_total / 1000:.0f}")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=aee5f11b-1a0f-4758-a796-2ef46a39caab
To: /content/data.parquet
100%|██████████| 130M/130M [00:01<00:00, 126MB/s]


El ingreso neto total obtenido es de: $66
